In [ ]:
#Loading all the necessary modules for the language model and chat model
from langchain_ollama import ChatOllama, OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

chat = ChatOllama(model="mistral")


Language Model - raw LLM-- takes a string in, returns a string


In [2]:
llm = OllamaLLM(model="mistral", temperature=0.2, max_tokens=512)
response = llm.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)

 There was no reported instance where OpenAI models attacked HuggingFace. Both are organizations that develop AI technologies and have collaborated on several projects, such as the release of the Codex model, which combines elements from both entities.


Chat Model- Layer over language model, built for multi-turn convesation with role-tagged messages; not flat messages.

In [3]:
chat = ChatOllama(model="mistral", temperature=0.2, max_tokens=512)
response = chat.invoke("When did OpenAI models attack HuggingFace in one sentence, accuracy is of importance")
print(response)


content=' There was no instance where OpenAI models attacked HuggingFace. Both are organizations that develop AI technologies and have collaborated on several projects, such as the release of the Codex model.' additional_kwargs={} response_metadata={'model': 'mistral', 'created_at': '2026-07-25T10:18:37.2057337Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6823665400, 'load_duration': 175036000, 'prompt_eval_count': 21, 'prompt_eval_duration': 257956000, 'eval_count': 39, 'eval_duration': 6367752000, 'logprobs': None, 'model_name': 'mistral', 'model_provider': 'ollama'} id='lc_run--019f98c8-9613-7be3-95c1-295f2bc723d4-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 21, 'output_tokens': 39, 'total_tokens': 60}


Chat message - role-tagged objects; what the above model consumes and produces  


In [6]:
messages = [
    SystemMessage(content="You are a an expert scientist that explains complex topics in simple terms."),
    HumanMessage(content="Explain the theory of relativity in simple terms."),
    AIMessage(content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes."),
    HumanMessage(content="Can you provide a simple analogy to help me understand it better?"),  
]
response = chat.invoke(messages)
print(response)
type(response)  # This will show the type of the response object

content=" Sure! Here's an analogy to help explain the theory of relativity:\n\nImagine space-time is like a rubber sheet stretched out flat. Now, if you place a heavy ball (like a planet) on this sheet, it will cause the sheet to curve around the ball. This curvature affects how other objects move on the sheet. For example, if you roll a marble near the ball, it will follow a curved path instead of moving in a straight line.\n\nThis is similar to what happens with gravity according to general relativity. Massive objects like planets and stars bend space-time around them, causing other objects to move along curved paths. This bending of space-time is what we perceive as gravity.\n\nIn special relativity, the analogy is a bit different. Imagine you are on a train moving at high speed relative to someone standing still on the platform. To you, everything inside the train appears normal, but to the person on the platform, the train seems to be moving and time inside the train seems to slow

langchain_core.messages.ai.AIMessage

Prompt templates - reusable!

In [9]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that translates English to French."),
    ("human", "Explain {topic} in simple terms."),
])
prompt = template.format_prompt(topic="the theory of relativity")
print(prompt.to_messages())


[SystemMessage(content='You are a helpful assistant that translates English to French.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain the theory of relativity in simple terms.', additional_kwargs={}, response_metadata={})]


Output parsers - model's raw text response is coerced into structured type to use with code directly

In [10]:
#PydanticOutputParser here returns a RAGAnswer- I need to read both in detail
parser = StrOutputParser()
chain = template | chat | parser
result = chain.invoke({"topic": "the theory of relativity"})
print(type(result))

<class 'langchain_core.messages.base.TextAccessor'>


Puting these steps together, we create chains as shown above. 
Documents: standard unit for a piece of retrievable text plus its metadata

In [13]:
doc = Document(
    page_content="The theory of relativity, developed by Albert Einstein, is a fundamental concept in physics that describes how space and time are interconnected. It consists of two main parts: special relativity and general relativity. Special relativity states that the laws of physics are the same for all observers, regardless of their relative motion, and it introduces the idea that time and space can stretch or contract depending on how fast you are moving. General relativity expands on this by explaining how gravity is not just a force between masses, but rather a curvature of space-time caused by mass and energy. In simple terms, it means that massive objects like planets and stars bend the fabric of space-time around them, which affects how objects move and how time passes.",
    metadata={"source": "https://en.wikipedia.org/wiki/Theory_of_relativity"}
)


Agents-chain runs a fixed sequence each time it runs

In [15]:
# from langchain_classic.agents import initialize_agent, Tool, AgentType

# def calculator(expr: str) -> str:
#     return str(eval(expr))

# tools = [Tool(name="Calculator", func=calculator, description="Evaluates math expressions")]

# agent = initialize_agent(tools, chat, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)
# result = agent.invoke("What is 47 * 12, and then explain the result in words?")
# print(result)